## Install OCI CLI (if not already installed)
```bash
bash -c "$(curl -L https://raw.githubusercontent.com/oracle/oci-cli/master/scripts/install/install.sh)"
```

In [ ]:
!which oci

In [ ]:
!oci session refresh --profile harleycalvert

# Set Up OCI Terraform

[Set Up OCI Terraform](https://docs.oracle.com/en-us/iaas/developer-tutorials/tutorials/tf-provider/01-summary.htm)

## Install OpenTofu

In [ ]:
!snap install --classic opentofu

In [ ]:
!tofu -v

## Create RSA Keys

In [ ]:
!mkdir $HOME/.oci

In [ ]:
!openssl genrsa -out $HOME/.oci/oci_rsa_key.pem 2048

In [ ]:
!chmod 600 $HOME/.oci/oci_rsa_key.pem

In [ ]:
!openssl rsa -pubout -in $HOME/.oci/oci_rsa_key.pem -out $HOME/.oci/oci_rsa_key.pem.pub

In [ ]:
!(cd $HOME/.oci/ && ls)

In [ ]:
!cat $HOME/.oci/oci_rsa_key.pem.pub

Add the public key to your user account.

In the OCI Console's top navigation bar, click the Profile menu, and then go to My profile.
- Click API Keys.
- Click Add API Key.
- Select Paste Public Keys.
- Paste value from previous step, including the lines with BEGIN PUBLIC KEY and END PUBLIC KEY.
- Click Add.
- Copy the Configuration file preview.
- Paste the Configuration file preview into ~/.oci/config

You have now set up the RSA keys to connect to your OCI account.

## Structure

adb-infra/
  main.tf
  variables.tf
  outputs.tf
  provider.tf
  /modules
    /adb
      main.tf
      variables.tf
      outputs.tf
    /compartment
      main.tf
      variables.tf
      outputs.tf
    /compute
      main.tf
      variables.tf
      outputs.tf
    /lb
      main.tf
      variables.tf
      outputs.tf
    /nsg
      main.tf
      variables.tf
      outputs.tf
    /vcn
      main.tf
      variables.tf
      outputs.tf


In [ ]:
!mkdir modules
!mkdir -p modules/adb modules/compartment modules/compute modules/lb modules/nsg modules/vcn
!tree

In [ ]:
%%writefile ./provider.tf

provider "oci" {
  config_file_profile = "harleycalvert"
}

# Main / Root

In [ ]:
%%writefile ./main.tf

module "compartment" {
  source           = "./modules/compartment"
  tenancy_id       = var.tenancy_id
  compartment_name = var.compartment_name
}

module "vcn" {
  source             = "./modules/vcn"
  compartment_id     = module.compartment.id
  vcn_cidr           = var.vcn_cidr
  public_subnet_cidr = var.public_subnet_cidr
  admin_ip           = var.admin_ip
}

module "nsg" {
  source         = "./modules/nsg"
  compartment_id = module.compartment.id
  vcn_id         = module.vcn.vcn_id
  admin_ip       = var.admin_ip
}

module "compute" {
  source   = "./modules/compute"
  for_each = toset(["vm-01", "vm-02"])

  tenancy_id          = var.tenancy_id
  compartment_id      = module.compartment.id
  subnet_id           = module.vcn.public_subnet_id
  nsg_ids             = [module.nsg.adb_nsg_id]
  assign_public_ip    = true
  ssh_public_key_path = var.ssh_public_key_path
  display_name        = each.key
  hostname            = each.key
  shape               = var.shape
  image_id            = var.image_id
}

locals {
  backend_ips = {
    for name, vm in module.compute :
    name => vm.private_ip
  }
}

module "lb" {
  source         = "./modules/lb"
  compartment_id = module.compartment.id
  subnet_id      = module.vcn.public_subnet_id
  nsg_ids        = [module.nsg.lb_nsg_id]
  backend_ips    = local.backend_ips

  ca_certificate_path     = var.ca_certificate_path
  public_certificate_path = var.public_certificate_path
  private_key_path        = var.private_key_path
}

module "adb" {
  source         = "./modules/adb"
  compartment_id = module.compartment.id
  db_name        = var.adb_db_name
  display_name   = var.adb_display_name
  admin_password = var.adb_admin_password
}


In [ ]:
%%writefile ./variables.tf

variable "tenancy_id" {
  type        = string
  description = "OCID of the tenancy"
}

variable "compartment_name" {
  type        = string
  description = "Name of the OCI compartment to create"
}

variable "vcn_cidr" {
  type        = string
  description = "CIDR block for the VCN"
}

variable "public_subnet_cidr" {
  type        = string
  description = "CIDR block for the public subnet"
}

variable "shape" {
  type        = string
  description = "Compute instance shape"
}

variable "image_id" {
  type        = string
  description = "OCID of the compute instance image"
}

variable "admin_ip" {
  type        = string
  description = "Admin public IP in CIDR format"
}

variable "ssh_public_key_path" {
  type        = string
  description = "Path to SSH public key for VM access"
}

variable "adb_db_name" {
  type        = string
  description = "ADB database name: alphanumeric, starts with letter, max 14 chars"
}

variable "adb_display_name" {
  type        = string
  description = "ADB display name shown in OCI console"
}

variable "adb_admin_password" {
  type        = string
  sensitive   = true
  description = "ADB ADMIN password: 12+ chars, upper, lower, number, special"
}

variable "ca_certificate_path" {
  type        = string
  description = "Path to CA certificate PEM file"
}

variable "public_certificate_path" {
  type        = string
  description = "Path to public certificate PEM file"
}

variable "private_key_path" {
  type        = string
  description = "Path to private key PEM file"
}


In [ ]:
%%writefile ./terraform.tfvars

# this file is not submitted to Git - specify in .gitignore

tenancy_id         = "<your-tenancy_id>"
compartment_name   = "adb-22.04"
vcn_cidr           = "10.0.0.0/16"
public_subnet_cidr = "10.0.0.0/24"
# https://docs.oracle.com/en-us/iaas/images/
# Use Canonical-Ubuntu-22.04 for your region
shape    = "VM.Standard.E2.1.Micro"
image_id = "<ocid1.image.oc1.ap-melbourne-1.xxxx>"
admin_ip = "<your-public-ip>/32"  #<YOUR_PUBLIC_IP>/32 - Frankston
ssh_public_key_path = "~/.ssh/adb_vm_key.pub"

adb_db_name        = "adb2204"
adb_display_name   = "adb-22.04"
adb_admin_password = "<top-secret-password>"  # replace before apply

ca_certificate_path     = "~/.tls/chain.pem"
public_certificate_path = "~/.tls/fullchain.pem"
private_key_path        = "~/.tls/privkey.pem"


In [ ]:
%%writefile ./outputs.tf

output "lb_public_ip" {
  value = module.lb.public_ip
}

output "vm_public_ips" {
  value = {
    for name, vm in module.compute :
    name => vm.public_ip
  }
}

output "adb_id" {
  value = module.adb.id
}

output "adb_service_console_url" {
  value = module.adb.service_console_url
}


# Compartment

In [ ]:
%%writefile ./modules/compartment/main.tf

resource "oci_identity_compartment" "compartment" {
  compartment_id = var.tenancy_id
  name           = var.compartment_name
  description    = "Managed by OpenTofu"
  enable_delete  = true
}


In [ ]:
%%writefile ./modules/compartment/variables.tf

variable "tenancy_id" {
  type = string
}

variable "compartment_name" {
  type = string
}


In [ ]:
%%writefile ./modules/compartment/outputs.tf

output "id" {
  value = oci_identity_compartment.compartment.id
}


# VCN

In [ ]:
%%writefile ./modules/vcn/main.tf

resource "oci_core_vcn" "vcn" {
  compartment_id = var.compartment_id
  cidr_block     = var.vcn_cidr
  display_name   = "vcn"
  dns_label = "vcn"
}

resource "oci_core_internet_gateway" "igw" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "igw"
  enabled        = true
}

resource "oci_core_route_table" "public_rt" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "public-rt"

  route_rules {
    destination       = "0.0.0.0/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_internet_gateway.igw.id
  }
}

resource "oci_core_subnet" "public_subnet" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  cidr_block     = var.public_subnet_cidr
  display_name   = "public-subnet"
  route_table_id = oci_core_route_table.public_rt.id
  dhcp_options_id = data.oci_core_dhcp_options.default.options[0].id
  dns_label      = "public"

  security_list_ids = [
    oci_core_security_list.public_sl.id
  ]
}

data "oci_core_dhcp_options" "default" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "Default DHCP Options for vcn"
}


In [ ]:
%%writefile modules/vcn/security_lists.tf

resource "oci_core_security_list" "public_sl" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "public-subnet-sl"

  # --- INGRESS RULES ---

  # Allow SSH from your admin IP
  ingress_security_rules {
    protocol = "6"
    source   = var.admin_ip

    tcp_options {
      min = 22
      max = 22
    }
  }

  # Allow HTTP from anywhere
  # ingress_security_rules {
  #   protocol = "6"
  #   source   = "0.0.0.0/0"

  #   tcp_options {
  #     min = 80
  #     max = 80
  #   }
  # }

  # Allow ICMP fragmentation-needed
  ingress_security_rules {
    protocol = "1"
    source   = "0.0.0.0/0"

    icmp_options {
      type = 3
      code = 4
    }
  }

  # --- EGRESS RULES ---

  egress_security_rules {
    protocol    = "all"
    destination = "0.0.0.0/0"
  }
}


In [ ]:
%%writefile ./modules/vcn/variables.tf

variable "compartment_id" {
  type = string
}

variable "vcn_cidr" {
  type = string
}

variable "public_subnet_cidr" {
  type = string
}

variable "admin_ip" {
  type = string
}

In [ ]:
%%writefile ./modules/vcn/outputs.tf

output "vcn_id" {
  value = oci_core_vcn.vcn.id
}

output "public_subnet_id" {
  value = oci_core_subnet.public_subnet.id
}

# NSG - Network Security Groups

In [ ]:
%%writefile ./modules/nsg/main.tf

resource "oci_core_network_security_group" "lb" {
  compartment_id = var.compartment_id
  vcn_id         = var.vcn_id
  display_name   = "lb-nsg"
}

resource "oci_core_network_security_group" "adb" {
  compartment_id = var.compartment_id
  vcn_id         = var.vcn_id
  display_name   = "adb-nsg"
}

# Allow public HTTP to VM NSG
resource "oci_core_network_security_group_security_rule" "adb_ingress_http" {
  network_security_group_id = oci_core_network_security_group.adb.id
  direction                 = "INGRESS"
  protocol                  = "6" # TCP
  source                    = "0.0.0.0/0"

  tcp_options {
    destination_port_range {
      min = 80
      max = 80
    }
  }
}

# Allow SSH from admin IP to VM NSG
resource "oci_core_network_security_group_security_rule" "adb_ingress_ssh" {
  network_security_group_id = oci_core_network_security_group.adb.id
  direction                 = "INGRESS"
  protocol                  = "6" # TCP
  source                    = var.admin_ip

  tcp_options {
    destination_port_range {
      min = 22
      max = 22
    }
  }
}

# Allow public HTTPS to LB NSG
resource "oci_core_network_security_group_security_rule" "lb_ingress_https" {
  network_security_group_id = oci_core_network_security_group.lb.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source                    = "0.0.0.0/0"

  tcp_options {
    destination_port_range {
      min = 443
      max = 443
    }
  }
}

# Allow public HTTP to LB NSG
resource "oci_core_network_security_group_security_rule" "lb_ingress_http" {
  network_security_group_id = oci_core_network_security_group.lb.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source                    = "0.0.0.0/0"

  tcp_options {
    destination_port_range {
      min = 80
      max = 80
    }
  }
}

# Allow LB -> VM traffic
resource "oci_core_network_security_group_security_rule" "adb_ingress_from_lb" {
  network_security_group_id = oci_core_network_security_group.adb.id
  direction                 = "INGRESS"
  protocol                  = "6" # TCP
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.lb.id

  tcp_options {
    destination_port_range {
      min = 80
      max = 80
    }
  }
}

# Allow all egress from VM NSG
resource "oci_core_network_security_group_security_rule" "adb_egress" {
  network_security_group_id = oci_core_network_security_group.adb.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination               = "0.0.0.0/0"
}

# Allow all egress from LB NSG
resource "oci_core_network_security_group_security_rule" "lb_egress" {
  network_security_group_id = oci_core_network_security_group.lb.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination               = "0.0.0.0/0"
}


In [ ]:
%%writefile ./modules/nsg/variables.tf

variable "compartment_id" {
  type = string
}

variable "vcn_id" {
  type = string
}

variable "admin_ip" {
  type        = string
  description = "Public IP allowed to SSH into the VM"
}


In [ ]:
%%writefile ./modules/nsg/outputs.tf


output "lb_nsg_id" {
  value = oci_core_network_security_group.lb.id
}

output "adb_nsg_id" {
  value = oci_core_network_security_group.adb.id
}


# Compute

In [ ]:
%%writefile ./modules/compute/main.tf

data "oci_identity_availability_domains" "ads" {
  compartment_id = var.tenancy_id
}

resource "oci_core_instance" "vm" {
  availability_domain = data.oci_identity_availability_domains.ads.availability_domains[var.ad_index].name
  compartment_id      = var.compartment_id
  display_name        = var.display_name
  shape               = var.shape

  source_details {
    source_type = "image"
    source_id   = var.image_id
  }

  create_vnic_details {
    subnet_id        = var.subnet_id
    nsg_ids          = var.nsg_ids
    assign_public_ip = var.assign_public_ip
    hostname_label   = replace(var.hostname, "-", "")
  }

  metadata = {
    ssh_authorized_keys = file(pathexpand(var.ssh_public_key_path))
  }

  preserve_boot_volume = false
}


In [ ]:
%%writefile ./modules/compute/variables.tf

variable "tenancy_id" {
  type = string
}

variable "compartment_id" {
  type = string
}

variable "subnet_id" {
  type = string
}

variable "nsg_ids" {
  type = list(string)
}

variable "ssh_public_key_path" {
  type = string
}

variable "display_name" {
  type = string
}

variable "hostname" {
  type = string
}

variable "shape" {
  type    = string
}

variable "image_id" {
  type = string
}

variable "assign_public_ip" {
  type    = bool
  default = true  # change to false, once fully configured and SSH is no longer needed.
}

variable "ad_index" {
  type    = number
  default = 0
}



In [ ]:
%%writefile ./modules/compute/outputs.tf

output "instance_id" {
  value = oci_core_instance.vm.id
}

output "public_ip" {
  value = oci_core_instance.vm.public_ip
}

output "private_ip" {
  value = oci_core_instance.vm.private_ip
}


# Load Balancer

In [ ]:
%%writefile ./modules/lb/main.tf

resource "oci_load_balancer_load_balancer" "lb" {
  compartment_id             = var.compartment_id
  display_name               = var.display_name
  shape                      = "flexible"
  subnet_ids                 = [var.subnet_id]
  network_security_group_ids = var.nsg_ids

  shape_details {
    minimum_bandwidth_in_mbps = 10
    maximum_bandwidth_in_mbps = 10
  }
}

resource "oci_load_balancer_backend_set" "bs" {
  name             = var.backend_set_name
  load_balancer_id = oci_load_balancer_load_balancer.lb.id
  policy           = "ROUND_ROBIN"

  health_checker {
    protocol = "HTTP"
    url_path = "/"
    port     = var.backend_port
  }
}

resource "oci_load_balancer_backend" "backend" {
  for_each = var.backend_ips

  load_balancer_id = oci_load_balancer_load_balancer.lb.id
  backendset_name  = oci_load_balancer_backend_set.bs.name
  ip_address       = each.value
  port             = var.backend_port
  weight           = 1
}

resource "oci_load_balancer_certificate" "cert" {
  load_balancer_id = oci_load_balancer_load_balancer.lb.id
  certificate_name = var.certificate_name

  ca_certificate     = file(pathexpand(var.ca_certificate_path))
  public_certificate = file(pathexpand(var.public_certificate_path))
  private_key        = file(pathexpand(var.private_key_path))

  lifecycle {
    create_before_destroy = true
  }
}

# resource "oci_load_balancer_listener" "http_listener" {
#   load_balancer_id         = oci_load_balancer_load_balancer.lb.id
#   name                     = "http-listener"
#   default_backend_set_name = oci_load_balancer_backend_set.bs.name
#   port                     = 80
#   protocol                 = "HTTP"
# }

resource "oci_load_balancer_listener" "https_listener" {
  load_balancer_id         = oci_load_balancer_load_balancer.lb.id
  name                     = "https-listener"
  default_backend_set_name = oci_load_balancer_backend_set.bs.name
  port                     = 443
  protocol                 = "HTTP" # OCI uses HTTP even for TLS termination

  ssl_configuration {
    certificate_name        = oci_load_balancer_certificate.cert.certificate_name
    verify_peer_certificate = false
    cipher_suite_name       = "oci-default-ssl-cipher-suite-v1"
    protocols               = ["TLSv1.2", "TLSv1.3"]
  }
}


In [ ]:
%%writefile ./modules/lb/variables.tf

variable "compartment_id" {
  type = string
}

variable "subnet_id" {
  type = string
}

variable "nsg_ids" {
  type = list(string)
}

variable "display_name" {
  type    = string
  default = "lb"
}

variable "backend_set_name" {
  type    = string
  default = "adb-backend-set"
}

variable "backend_port" {
  type    = number
  default = 80
}

variable "backend_ips" {
  type        = map(string)
  description = "Map of VM name to private IP for LB backends"
}

variable "certificate_name" {
  type    = string
  default = "adb-cert"
}

variable "ca_certificate_path" {
  type        = string
  description = "Path to CA certificate PEM file"
}

variable "public_certificate_path" {
  type        = string
  description = "Path to public certificate PEM file"
}

variable "private_key_path" {
  type        = string
  description = "Path to private key PEM file"
}


In [ ]:
%%writefile ./modules/lb/outputs.tf

output "public_ip" {
  value = oci_load_balancer_load_balancer.lb.ip_address_details[0].ip_address
}


# ADB - Autonomous Database

Free-tier ATP (Autonomous Transaction Processing): 1 OCPU, 20 GB storage.

In [ ]:
%%writefile ./modules/adb/main.tf

resource "oci_database_autonomous_database" "adb" {
  compartment_id           = var.compartment_id
  db_name                  = var.db_name
  display_name             = var.display_name
  admin_password           = var.admin_password
  db_workload              = "OLTP"
  is_free_tier             = true
  cpu_core_count           = 1
  data_storage_size_in_tbs = 1
}


In [ ]:
%%writefile ./modules/adb/variables.tf

variable "compartment_id" {
  type = string
}

variable "db_name" {
  type        = string
  description = "Database name: alphanumeric, starts with letter, max 14 chars"
}

variable "display_name" {
  type = string
}

variable "admin_password" {
  type      = string
  sensitive = true
}


In [ ]:
%%writefile ./modules/adb/outputs.tf

output "id" {
  value = oci_database_autonomous_database.adb.id
}

output "service_console_url" {
  value = oci_database_autonomous_database.adb.service_console_url
}

output "connection_strings" {
  value     = oci_database_autonomous_database.adb.connection_strings
  sensitive = true
}


In [ ]:
!tree -al

# Create SSH keys - Run this in terminal
ssh-keygen -t rsa -b 4096 -f ~/.ssh/adb_vm_key

# Get a free domain
https://dash.int.yt/


# Generate a TLS cert
sudo snap install core

sudo snap refresh core

sudo snap install --classic certbot

sudo ln -s /snap/bin/certbot /usr/bin/certbot

certbot --version

sudo certbot certonly --standalone -d yourdomain.com -d www.yourdomain.com


In [ ]:
!tofu init

In [ ]:
!tofu validate

In [ ]:
!tofu plan

In [ ]:
!tofu apply -auto-approve

In [ ]:
!tofu output

## Connect to the ADB Instances
To connect to ADB1 VM:
```bash
ssh -i <ssh-private-key-path> ubuntu@<your-public-ip-address>

ssh -i /home/harley/.ssh/adb_vm_key ubuntu@207.211.157.172
# my old password

sudo apt update
sudo apt upgrade

```

## Set up Apache

[Install Apache and PHP on an Oracle Linux Instance](https://docs.oracle.com/en-us/iaas/developer-tutorials/tutorials/apache-on-oracle-linux/01-summary.htm)

Run the following command in the webserver terminal:

```bash
sudo su

apt update
apt -y install apache2
systemctl restart apache2

iptables -I INPUT 1 -m state --state NEW -p tcp --dport 80 -j ACCEPT
iptables -I INPUT 1 -m state --state NEW -p tcp --dport 443 -j ACCEPT
netfilter-persistent save
```

In the browser, try both vm_public_ips http://your-public-ip-address

Try the domain name https://your-domain.com

## Set up PHP

Run the following command in the webserver terminal:

```bash
apt -y install php libapache2-mod-php
php -v
systemctl restart apache2

echo '<?php
  phpinfo();
?>' > /var/www/html/info.php
```

In the browser, connect to http://your-public-ip-address/info.php

## Connecting PHP to Oracle Database

[Oracle Instant Client Downloads for Linux x86-64 (64-bit)
](https://www.oracle.com/database/technologies/instant-client/linux-x86-64-downloads.html)

Run the following command in the webserver terminal:

```bash
sudo su
mkdir /opt/oracle
cd /opt/oracle 

wget https://download.oracle.com/otn_software/linux/instantclient/2326200v2/instantclient-basic-linux.x64-23.26.2.0.0.zip
wget https://download.oracle.com/otn_software/linux/instantclient/2326200v2/instantclient-sdk-linux.x64-23.26.2.0.0.zip
    
apt-get install unzip  

unzip instantclient-basic-linux.x64-23.26.2.0.0.zip
unzip instantclient-sdk-linux.x64-23.26.2.0.0.zip

sh -c "echo /opt/oracle/instantclient_23_26 > \
      /etc/ld.so.conf.d/oracle-instantclient.conf"

ldconfig

apt install php-pear
apt install -y php-dev
```
[PECL](https://pecl.php.net/package/oci8)

```bash
pecl install oci8-3.2.1

'instantclient,/opt/oracle/instantclient_23_26'
```


If you get error "Use PHP OCI8..." Check here:

[oci8 3.2.0](https://pecl.php.net/package/oci8/3.2.0)

```bash
cd /usr/lib/php/20210902/

echo "extension = oci8.so" >> /etc/php/8.1/cli/php.ini
echo "extension = oci8.so" >> /etc/php/8.1/apache2/php.ini
```

Now reboot your webserver instance:

```bash
reboot
```

Check info.php in your browser for OCI configuration details.

In the Oracle Cloud console, go to your Database details -->> Database connection and download your wallet.

Then on your Ubuntu client extract your wallet and change into the wallet directory.

Edit sqlnet.ora

The text in the file should look like this:

```
WALLET_LOCATION = (SOURCE = (METHOD = file) (METHOD_DATA = (DIRECTORY="/opt/wallet")))
SSL_SERVER_DN_MATCH=yes
```

```bash
# IMPORTANT! Make sure you are in your wallet directory on your client
sftp -i <ssh-private-key-path> ubuntu@<your-public-ip-address>
mkdir wallet
cd wallet
put -r .
exit
```

```bash
ssh -i <ssh-private-key-path> ubuntu@<your-public-ip-address>
sudo su
mv wallet /opt/
exit
```

Copy the PHP code below into a file oci8.php updating your credentials and database address:

```php
<?php

error_reporting(E_ALL);
ini_set('display_errors', 'On');

$conn = oci_connect('USERNAME', 'PASSWORD', 'tcps://adb.ap-melbourne-1.oraclecloud.com:1522/*******_harleys_high.adb.oraclecloud.com?wallet_location=/opt/wallet');

if (!$conn) {
    $e = oci_error();
    trigger_error(htmlentities($e['message'] ?? ''), E_USER_ERROR);
}
var_dump($conn);


// Parse the statement. Note there is no final semi-colon in the SQL statement
// https://www.php.net/manual/en/function.oci-parse.php
$stid = oci_parse($conn, 'SELECT * FROM ADMIN.ACCOUNTS');
if (!$stid) {
    $e = oci_error($conn);
    trigger_error(htmlentities($e['message'] ?? ''), E_USER_ERROR);
}
var_dump($stid);


// Define variables to store column values
oci_define_by_name($stid, 'USER_ID', $column1);
oci_define_by_name($stid, 'EMAIL', $column2);
oci_define_by_name($stid, 'PASSWORD', $column3);

oci_execute($stid);

// https://www.php.net/manual/en/function.oci-fetch-array.php
// Fetch and display results
echo "<table border='1'>\n";
while (oci_fetch($stid)) {
    echo "<tr>\n";
    echo "    <td>" . htmlentities($column1 ?? '') . "</td>\n";
    echo "    <td>" . htmlentities($column2 ?? '') . "</td>\n";
    echo "    <td>" . htmlentities($column3 ?? '') . "</td>\n";
    echo "</tr>\n";
}
echo "</table>\n";

oci_free_statement($stid);
oci_close($conn);

?>
```

```bash
sftp -i <ssh-private-key-path> ubuntu@<your-public-ip-address>
put oci8.php
exit
```

```bash
sudo su
chown root:root oci8.php
chmod 644 oci8.php
mv oci8.php /var/www/html/
exit
```

Using the cloud console, go to Database Actions -> SQL and run:

```
CREATE TABLE ADMIN.ACCOUNTS (
    USER_ID   NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    EMAIL     VARCHAR2(255) NOT NULL,
    PASSWORD  VARCHAR2(255) NOT NULL
);
COMMIT;

INSERT INTO ADMIN.ACCOUNTS (EMAIL, PASSWORD) VALUES ('alice@example.com', 'hash1');
COMMIT;

INSERT INTO ADMIN.ACCOUNTS (EMAIL, PASSWORD) VALUES ('bob@example.com',   'hash2');
COMMIT;

INSERT INTO ADMIN.ACCOUNTS (EMAIL, PASSWORD) VALUES ('your_name@example.com',   'hash3');
COMMIT;


Now visit https://<your-domain-name>/oci8.php
```

# Destroy Resources

In [ ]:
!tofu destroy -auto-approve